In [66]:
import pandas as pd
from sklearn.linear_model import LinearRegression
import numpy as np

In [68]:
# 1. Dosyayı yükle
veriler_path = 'tahminsonuclari.csv'  # 2002-2024 arası veriler
veriler = pd.read_csv(veriler_path)

In [70]:
# 2. Kullanılacak değişkenler
hedef_degiskenler = [
    "nufus", "hastane", "yatak",
    "uzman", "asistan", "pratisyen", 
    "diger", "hemsire", "ebe", "disci", "eczaci"
]

In [72]:
# 3. 2002-2024 verilerini seç
veriler_2002_2024 = veriler[veriler['yıl'] <= 2024]

In [74]:
# 4. 2025 için tahmin yapacak model
yeni_veri_2025 = {'yıl': [2025]}
for degisken in hedef_degiskenler:
    X = veriler_2002_2024[['yıl']]
    y = veriler_2002_2024[degisken]
    
    model = LinearRegression()
    model.fit(X, y)
    
    tahmin_2025 = model.predict(pd.DataFrame({'yıl': [2025]}))[0]
    
    # Negatif olmasın
    tahmin_2025 = max(tahmin_2025, 0)
    yeni_veri_2025[degisken] = [tahmin_2025]

In [76]:
# 5. 2025 verisini DataFrame haline getir
veri_2025_df = pd.DataFrame(yeni_veri_2025)

In [78]:
# 6. Şehir bazlı veri oluşturmak için şehirleri al
sehirler = veriler_2002_2024['il_adı'].str.lower().unique()

In [80]:
# 7. 2025 verisini şehir bazlı oluştur
veriler_2025 = pd.DataFrame({'il_adı': sehirler})
for kolon in hedef_degiskenler:
    toplam_2024 = veriler_2002_2024[veriler_2002_2024['yıl'] == 2024][kolon].sum()
    
    if toplam_2024 == 0:  # Sıfır bölme hatası olmasın
        veriler_2025[kolon] = 0
    else:
        veriler_2025[kolon] = (
            veriler_2002_2024[veriler_2002_2024['yıl'] == 2024][['il_adı', kolon]]
            .assign(il_adı=lambda x: x['il_adı'].str.lower())  # şehir adını küçült
            .set_index('il_adı')
            .loc[sehirler][kolon]
            / toplam_2024
        ) * yeni_veri_2025[kolon][0]

In [82]:
# 8. 2025 yılı bilgisi ekle
veriler_2025['yıl'] = 2025

In [84]:
# 9. Personel/Yatak ve Personel/Hastane oranlarını hesapla
personel_turleri = [
    "uzman", "asistan", "pratisyen", 
    "diger", "hemsire", "ebe", "disci", "eczaci"
]

for personel in personel_turleri:
    veriler_2025[f"{personel}_per_yatak"] = veriler_2025[personel] / veriler_2025["yatak"]
    veriler_2025[f"{personel}_per_hastane"] = veriler_2025[personel] / veriler_2025["hastane"]

In [86]:
# 10. Sonuçları CSV olarak kaydet
veriler_2025.to_csv('2025_tahmini_sehir_verileri.csv', index=False, float_format='%.8f')

print("✅ 2025 tahmini şehir bazlı veriler çıkarıldı ve '/mnt/data/2025_tahmini_sehir_verileri.csv' olarak kaydedildi.")

✅ 2025 tahmini şehir bazlı veriler çıkarıldı ve '/mnt/data/2025_tahmini_sehir_verileri.csv' olarak kaydedildi.
